# Stage 1: Dataset Inspection
**Deep Learning-Based Detection and Classification of Brain Tumors Using MRI Images**

Nothing in this notebook is assumed about the dataset. Every number is computed from the downloaded files.
Run the cells in order. Runtime: GPU (not needed for Stage 1, but keeps the environment identical for later stages).

## 1. Setup, seeds and environment record

In [ ]:
import os, sys, json, random, hashlib, re, zipfile, platform, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import torch, torchvision
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

PROJECT = Path('/content/brain-tumor-project')
for d in ['data','notebooks','src','models','results/figures','results/metrics','results/predictions']:
    (PROJECT/d).mkdir(parents=True, exist_ok=True)
RAW = PROJECT/'data'/'raw'; RAW.mkdir(exist_ok=True)

env = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
           cuda_available=torch.cuda.is_available(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, seed=SEED)
print(json.dumps(env, indent=2))

## 2. Kaggle credentials and download
No credentials are hard-coded. Choose ONE option:

**Option A (recommended): Colab Secrets.** Kaggle > Settings > API > Create New Token (downloads `kaggle.json`). In Colab click the key icon (left sidebar), add secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY` with the values from that file, and enable "Notebook access".

**Option B:** the cell will ask you to upload `kaggle.json` if the secrets are not found.

You may also need to open the dataset page once on Kaggle and accept its terms while logged in.

In [ ]:
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'

def setup_kaggle():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'environment variables'
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        return 'Colab secrets'
    except Exception:
        from google.colab import files
        print('Colab secrets not found. Please upload your kaggle.json')
        up = files.upload()
        kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
        (kdir/'kaggle.json').write_bytes(up[list(up)[0]])
        os.chmod(kdir/'kaggle.json', 0o600)
        return 'uploaded kaggle.json'

already = any(p.is_file() for p in RAW.rglob('*'))
if already:
    print('Data already present in', RAW, '- skipping download.')
else:
    print('Credentials from:', setup_kaggle())
    subprocess.run(['pip','-q','install','kaggle'], check=True)
    subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], check=True)

# Extract any nested zip files that remain
for z in list(RAW.rglob('*.zip')):
    print('Extracting nested archive:', z)
    with zipfile.ZipFile(z) as f: f.extractall(z.parent/z.stem)

n_files = sum(1 for p in RAW.rglob('*') if p.is_file())
print('Files found under', RAW, ':', n_files)
assert n_files > 0, 'No files found. Check the Kaggle download output above.'

## 3. Folder structure (discovered, not assumed)

In [ ]:
def count_files(d): return sum(1 for p in d.rglob('*') if p.is_file())

def show_tree(root, max_depth=4, max_dirs=15, max_files=4, indent=''):
    subdirs = sorted([p for p in root.iterdir() if p.is_dir()])
    files = sorted([p for p in root.iterdir() if p.is_file()])
    for d in subdirs[:max_dirs]:
        print(f'{indent}[D] {d.name}/  ({count_files(d)} files)')
        if max_depth > 1: show_tree(d, max_depth-1, max_dirs, max_files, indent+'    ')
    if len(subdirs) > max_dirs: print(f'{indent}... {len(subdirs)-max_dirs} more folders')
    for f in files[:max_files]: print(f'{indent}- {f.name}')
    if len(files) > max_files: print(f'{indent}... {len(files)-max_files} more files in this folder')

print(RAW.name + '/'); show_tree(RAW)

## 4. File inventory

In [ ]:
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
ANN_EXT = {'.xml','.json','.txt','.csv','.yaml','.yml'}

inv = pd.DataFrame([{'path': p, 'rel': str(p.relative_to(RAW)), 'name': p.name, 'stem': p.stem,
                     'ext': p.suffix.lower(), 'size_kb': p.stat().st_size/1024}
                    for p in RAW.rglob('*') if p.is_file()])
inv['is_image'] = inv.ext.isin(IMG_EXT)
print('Total files:', len(inv))
print('\nFiles by extension:'); print(inv.ext.value_counts().to_string())
print('\nTotal image files:', int(inv.is_image.sum()))
stem_dups = inv[inv.is_image].stem.duplicated(keep=False).sum()
print('Image files sharing a filename stem with another image (matters for annotation matching):', int(stem_dups))

## 5. Image properties and integrity (every image is opened)

In [ ]:
rows, bad = [], []
for p in tqdm(inv[inv.is_image].path.tolist()):
    try:
        with Image.open(p) as im: im.verify()            # structural check
        with Image.open(p) as im:
            fmt, mode, (w, h), nb = im.format, im.mode, im.size, len(im.getbands())
            arr = np.asarray(im.convert('RGB'))
        gray_like = bool(np.array_equal(arr[...,0], arr[...,1]) and np.array_equal(arr[...,1], arr[...,2]))
        rows.append(dict(path=p, format=fmt, mode=mode, width=w, height=h, n_bands=nb,
                         gray_like=gray_like, mean=float(arr.mean()), std=float(arr.std()),
                         min=int(arr.min()), max=int(arr.max())))
    except Exception as e:
        bad.append((str(p), repr(e)))

props = pd.DataFrame(rows)
img_df = inv[inv.is_image].merge(props, on='path', how='inner').reset_index(drop=True)
print('Readable images:', len(img_df), '| Corrupted/unreadable:', len(bad))
for b in bad[:20]: print('  BAD:', b)
print('\nFormats:\n', img_df.format.value_counts().to_string())
print('\nColour modes (PIL):\n', img_df['mode'].value_counts().to_string())
print('\nNumber of bands:\n', img_df.n_bands.value_counts().to_string())
print('\nRGB-stored images whose 3 channels are identical (i.e. effectively grayscale):',
      int(((img_df['mode'].isin(['RGB','RGBA'])) & img_df.gray_like).sum()))
img_df['size'] = img_df.width.astype(str) + 'x' + img_df.height.astype(str)
print('\nDistinct image sizes:', img_df['size'].nunique())
print(img_df['size'].value_counts().head(15).to_string())
print('\nWidth  min/median/max:', img_df.width.min(), img_df.width.median(), img_df.width.max())
print('Height min/median/max:', img_df.height.min(), img_df.height.median(), img_df.height.max())
print('Pixel mean/std (global):', round(img_df['mean'].mean(),2), round(img_df['std'].mean(),2))
print('Images that look almost blank (std < 5):', int((img_df['std'] < 5).sum()))

## 6. Existing splits and class labels
Class labels are *inferred* from the nearest meaningful parent folder. **Read the printout carefully.** If the labels come from somewhere else (for example only from annotation files), the next stage will adapt.

In [ ]:
SPLIT_TOKENS = {'train':'train','training':'train','val':'val','valid':'val','validation':'val',
                'test':'test','testing':'test'}
GENERIC = set(SPLIT_TOKENS) | {'images','image','img','imgs','labels','label','annotations','annotation',
           'data','raw','jpg','png','dataset','mri','all'}

def infer_split(rel):
    for part in Path(rel).parts[:-1]:
        if part.lower() in SPLIT_TOKENS: return SPLIT_TOKENS[part.lower()]
    return None

def infer_label(rel):
    for part in reversed(Path(rel).parts[:-1]):
        if part.lower() not in GENERIC: return part
    return None

img_df['split'] = img_df.rel.map(infer_split)
img_df['label_guess'] = img_df.rel.map(infer_label)
print('Existing split folders detected:', sorted(img_df.split.dropna().unique()) or 'none')
print('\nCandidate class folders (label_guess):'); print(img_df.label_guess.value_counts(dropna=False).to_string())
if img_df.split.notna().any():
    print('\nSplit x class table:')
    print(pd.crosstab(img_df.split.fillna('none'), img_df.label_guess.fillna('none')))
print('\nNumber of candidate classes:', img_df.label_guess.nunique())

## 7. Annotation files
Each annotation file is parsed according to what it actually contains (Pascal VOC XML, COCO JSON, YOLO txt, or CSV). A raw sample of every type is printed first so you can verify the format by eye.

In [ ]:
import xml.etree.ElementTree as ET
ann_inv = inv[inv.ext.isin(ANN_EXT)].copy()
print('Candidate annotation/text files by extension:'); print(ann_inv.ext.value_counts().to_string())

# raw samples, one per extension
for ext, grp in ann_inv.groupby('ext'):
    p = grp.path.iloc[0]
    print(f'\n--- raw sample ({ext}): {p.relative_to(RAW)} ---')
    try: print(p.read_text(errors='replace')[:600])
    except Exception as e: print('cannot read:', e)

In [ ]:
records, ann_files_seen, unparsed = [], {}, []

def parse_voc(p):
    r = ET.parse(p).getroot(); sz = r.find('size')
    W = int(float(sz.findtext('width'))) if sz is not None and sz.findtext('width') else None
    H = int(float(sz.findtext('height'))) if sz is not None and sz.findtext('height') else None
    key = Path(r.findtext('filename')).stem if r.findtext('filename') else p.stem
    out = []
    for o in r.findall('object'):
        b = o.find('bndbox')
        out.append(dict(key=key, cls=o.findtext('name'), fmt='voc_xyxy_abs', W=W, H=H,
                        v1=float(b.findtext('xmin')), v2=float(b.findtext('ymin')),
                        v3=float(b.findtext('xmax')), v4=float(b.findtext('ymax'))))
    return key, out

def parse_coco(p):
    d = json.load(open(p)); cats = {c['id']: c['name'] for c in d.get('categories', [])}
    ims = {i['id']: i for i in d.get('images', [])}; out = []
    for a in d.get('annotations', []):
        im = ims.get(a.get('image_id'), {}); x, y, w, h = a['bbox']
        out.append(dict(key=Path(im.get('file_name','')).stem, cls=cats.get(a.get('category_id'), a.get('category_id')),
                        fmt='coco_xywh_abs', W=im.get('width'), H=im.get('height'), v1=x, v2=y, v3=w, v4=h))
    return None, out

def parse_yolo(p):
    out = []
    lines = [l.split() for l in p.read_text().strip().splitlines() if l.strip()]
    for t in lines:
        vals = [float(v) for v in t[:5]]   # raises if not numeric -> not YOLO
        if len(t) < 5: raise ValueError('not yolo')
        out.append(dict(key=p.stem, cls=int(vals[0]), fmt='yolo_cxcywh_norm', W=None, H=None,
                        v1=vals[1], v2=vals[2], v3=vals[3], v4=vals[4]))
    return p.stem, out

for p in tqdm(ann_inv.path.tolist()):
    try:
        if p.suffix.lower() == '.xml':
            k, out = parse_voc(p)
        elif p.suffix.lower() == '.json':
            k, out = parse_coco(p)
        elif p.suffix.lower() == '.txt':
            k, out = parse_yolo(p)
        else:
            unparsed.append(p); continue
        records += out
        if k is not None: ann_files_seen[k] = ann_files_seen.get(k, 0) + len(out)
    except Exception:
        unparsed.append(p)

ann = pd.DataFrame(records)
print('Parsed bounding boxes:', len(ann), '| files that could not be parsed by the built-in parsers:', len(unparsed))
if unparsed:
    print('Unparsed examples (CSV or other formats need a custom parser, tell me what these are):')
    for u in unparsed[:10]: print('  ', u.relative_to(RAW))

In [ ]:
# CSV annotations (only handled automatically when column names are recognisable)
csvs = [p for p in unparsed if p.suffix.lower() == '.csv']
for p in csvs[:5]:
    d = pd.read_csv(p)
    print(f'\n{p.relative_to(RAW)}  shape={d.shape}'); print(d.head(5).to_string()); print('columns:', list(d.columns))

In [ ]:
if len(ann):
    print('Detected bounding-box format(s):', ann.fmt.value_counts().to_dict())
    print('\nBox classes:', ann.cls.value_counts(dropna=False).to_dict())
    for f, g in ann.groupby('fmt'):
        print(f'\nCoordinate ranges for {f}:'); print(g[['v1','v2','v3','v4']].describe().loc[['min','max','mean']].round(3).to_string())
        if f == 'voc_xyxy_abs':
            print('  boxes with xmax<=xmin or ymax<=ymin:', int(((g.v3<=g.v1)|(g.v4<=g.v2)).sum()))
        if f == 'yolo_cxcywh_norm':
            print('  values outside [0,1]:', int(((g[['v1','v2','v3','v4']]<0)|(g[['v1','v2','v3','v4']]>1)).any(axis=1).sum()))

    # link annotations to images by filename stem
    img_stems = img_df.groupby('stem').path.apply(list)
    ann_keys = set(ann.key.dropna())
    img_df['n_boxes'] = img_df.stem.map(lambda s: ann_files_seen.get(s, 0))
    img_df['has_ann_file'] = img_df.stem.isin(ann_files_seen)
    print('\nImages with an annotation file:', int(img_df.has_ann_file.sum()), 'of', len(img_df))
    print('Images with >=1 bounding box:', int((img_df.n_boxes>0).sum()))
    print('Annotation keys with no matching image:', len(set(ann_files_seen) - set(img_stems.index)))
    print('Boxes per image distribution (images with boxes):'); print(img_df[img_df.n_boxes>0].n_boxes.value_counts().sort_index().to_string())
    print('\nBounding boxes by class folder (label_guess):')
    print(pd.crosstab(img_df.label_guess.fillna('none'), img_df.n_boxes>0, rownames=['class'], colnames=['has_box']))
else:
    print('No bounding boxes were parsed automatically. Inspect the raw samples above and report back.')

## 8. Duplicate and near-duplicate detection
* **Exact duplicates**: identical file bytes (MD5).
* **Near duplicates**: 64-bit difference hash (dHash) of the grayscale image; pairs with Hamming distance <= `NEAR_THR` are *candidates*. This is a heuristic, so candidates are shown visually for manual confirmation rather than being deleted automatically.

In [ ]:
NEAR_THR = 4
def md5(p):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def dhash(p):
    im = Image.open(p).convert('L').resize((9,8), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16); bits = (a[:,1:] > a[:,:-1]).flatten()
    return int(np.packbits(bits).view('>u8')[0])

img_df['md5'] = [md5(p) for p in tqdm(img_df.path, desc='md5')]
img_df['dhash'] = [dhash(p) for p in tqdm(img_df.path, desc='dhash')]

exact = img_df[img_df.md5.duplicated(keep=False)].sort_values('md5')
print('Images involved in exact-duplicate groups:', len(exact), '| groups:', exact.md5.nunique())
if len(exact):
    g = exact.groupby('md5').agg(n=('path','size'), labels=('label_guess', lambda s: sorted(set(map(str,s)))),
                                  splits=('split', lambda s: sorted(set(map(str,s)))))
    print('Exact-duplicate groups spanning >1 label  :', int((g.labels.str.len()>1).sum()))
    print('Exact-duplicate groups spanning >1 split  :', int((g.splits.str.len()>1).sum()))
    print(g.head(10).to_string())

# near-duplicates by pairwise Hamming distance (chunked, numpy popcount lookup)
H = img_df.dhash.values.astype(np.uint64); N = len(H)
pop = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
pairs = []
for i in tqdm(range(0, N, 500), desc='near-dup'):
    x = H[i:i+500, None] ^ H[None, :]
    d = pop[x.view(np.uint8).reshape(x.shape[0], x.shape[1], 8)].sum(axis=2)
    ii, jj = np.where(d <= NEAR_THR)
    for a, b in zip(ii + i, jj):
        if a < b: pairs.append((int(a), int(b), int(d[a - i, b])))
pairs = pd.DataFrame(pairs, columns=['i','j','dist'])
pairs['same_md5'] = img_df.md5.values[pairs.i] == img_df.md5.values[pairs.j]
near_only = pairs[~pairs.same_md5]
print(f'\nPairs with dHash distance <= {NEAR_THR}: {len(pairs)} (exact-byte duplicates: {int(pairs.same_md5.sum())}, near-only: {len(near_only)})')
if len(near_only):
    li, lj = img_df.label_guess.values[near_only.i], img_df.label_guess.values[near_only.j]
    print('Near-duplicate candidate pairs with DIFFERENT class labels:', int((li != lj).sum()))

In [ ]:
# visual check of up to 6 near-duplicate candidate pairs
show = near_only.head(6)
if len(show):
    fig, ax = plt.subplots(len(show), 2, figsize=(6, 3*len(show)), squeeze=False)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, idx in enumerate([row.i, row.j]):
            ax[r][c].imshow(Image.open(img_df.path[idx]).convert('L'), cmap='gray'); ax[r][c].axis('off')
            ax[r][c].set_title(f'{img_df.label_guess[idx]} | d={row.dist}', fontsize=8)
    plt.tight_layout(); plt.savefig(PROJECT/'results/figures/near_duplicate_candidates.png', dpi=150); plt.show()
else:
    print('No near-duplicate candidate pairs to display.')

## 9. Patient identifiers and grouping information
Looks for patient/subject IDs in filenames, folder names and image metadata. Absence of evidence is reported as "not found", which is not the same as "images are from different patients".

In [ ]:
print('Example filenames:'); print(img_df.name.sample(min(15, len(img_df)), random_state=SEED).to_string(index=False))
pattern = img_df.stem.str.replace(r'\d+', '#', regex=True)
print('\nFilename patterns (digits replaced by #), top 15:'); print(pattern.value_counts().head(15).to_string())

# metadata fields embedded in the images
meta_keys = Counter()
for p in img_df.path.sample(min(300, len(img_df)), random_state=SEED):
    with Image.open(p) as im:
        meta_keys.update(im.info.keys()); 
        if hasattr(im, 'getexif'): meta_keys.update([f'exif:{k}' for k in im.getexif().keys()])
print('\nMetadata keys found in a sample of images:', dict(meta_keys) or 'none')

# any non-image, non-annotation tabular/text file that may hold patient info?
others = inv[~inv.is_image & ~inv.ext.isin({'.xml','.json'})]
print('\nOther files (check for README / metadata / labels tables):'); print(others[['rel','size_kb']].head(20).to_string(index=False))

## 10. Report-ready figures

In [ ]:
def box_xyxy(r, W, H):
    if r.fmt == 'voc_xyxy_abs':  return r.v1, r.v2, r.v3, r.v4
    if r.fmt == 'coco_xywh_abs': return r.v1, r.v2, r.v1 + r.v3, r.v2 + r.v4
    if r.fmt == 'yolo_cxcywh_norm':
        return (r.v1-r.v3/2)*W, (r.v2-r.v4/2)*H, (r.v1+r.v3/2)*W, (r.v2+r.v4/2)*H

# 1) class distribution (uses label_guess; confirm it is correct for this dataset)
if img_df.label_guess.notna().any():
    order = img_df.label_guess.value_counts().index
    plt.figure(figsize=(7,4)); ax = sns.countplot(data=img_df, x='label_guess', order=order, color='steelblue')
    for c in ax.containers: ax.bar_label(c)
    plt.title('Number of images per class (all files)'); plt.xlabel('Class'); plt.ylabel('Images'); plt.xticks(rotation=20)
    plt.tight_layout(); plt.savefig(PROJECT/'results/figures/class_distribution.png', dpi=200); plt.show()

# 2) sample images per class, with bounding boxes overlaid where available
classes = [c for c in img_df.label_guess.value_counts().index if c is not None][:6]
if classes:
    n = 4; fig, axes = plt.subplots(len(classes), n, figsize=(3*n, 3*len(classes)), squeeze=False)
    for r, c in enumerate(classes):
        sub = img_df[img_df.label_guess == c].sample(min(n, (img_df.label_guess == c).sum()), random_state=SEED)
        for k in range(n):
            axes[r][k].axis('off')
            if k >= len(sub): continue
            row = sub.iloc[k]; axes[r][k].imshow(Image.open(row.path).convert('L'), cmap='gray')
            if len(ann) and row.n_boxes > 0:
                for _, b in ann[ann.key == row.stem].iterrows():
                    x1, y1, x2, y2 = box_xyxy(b, row.width, row.height)
                    axes[r][k].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', lw=1.5))
            axes[r][k].set_title(f'{c} ({row.width}x{row.height})', fontsize=8)
    plt.suptitle('Random samples per class (red = annotated box)'); plt.tight_layout()
    plt.savefig(PROJECT/'results/figures/sample_images_per_class.png', dpi=200); plt.show()

# 3) image size and intensity overview
fig, ax = plt.subplots(1, 2, figsize=(10,3.5))
ax[0].hist(img_df.width, bins=30, alpha=.7, label='width'); ax[0].hist(img_df.height, bins=30, alpha=.7, label='height'); ax[0].legend(); ax[0].set_title('Image dimensions (px)')
ax[1].hist(img_df['mean'], bins=30, color='gray'); ax[1].set_title('Mean pixel intensity per image')
plt.tight_layout(); plt.savefig(PROJECT/'results/figures/image_size_intensity.png', dpi=200); plt.show()

## 11. Save the inspection results

In [ ]:
out = img_df.drop(columns=['path']).assign(path=img_df.path.astype(str))
out.to_csv(PROJECT/'results/metrics/image_inventory.csv', index=False)
if len(ann): ann.to_csv(PROJECT/'results/metrics/annotations_parsed.csv', index=False)

summary = dict(
    environment=env, dataset_slug=SLUG, total_files=int(len(inv)), image_files=int(inv.is_image.sum()),
    readable_images=int(len(img_df)), corrupted_images=len(bad),
    extensions=inv.ext.value_counts().to_dict(),
    candidate_classes=img_df.label_guess.value_counts(dropna=False).rename(index=str).to_dict(),
    existing_splits=sorted(img_df.split.dropna().unique().tolist()),
    distinct_image_sizes=int(img_df['size'].nunique()), modes=img_df['mode'].value_counts().to_dict(),
    exact_duplicate_groups=int(exact.md5.nunique()) if len(exact) else 0,
    near_duplicate_candidate_pairs=int(len(near_only)),
    boxes_parsed=int(len(ann)), box_formats=ann.fmt.value_counts().to_dict() if len(ann) else {},
    images_with_boxes=int((img_df.n_boxes>0).sum()) if 'n_boxes' in img_df else 0)
json.dump(summary, open(PROJECT/'results/metrics/stage1_summary.json','w'), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))

## What to send back
Copy the printed outputs of sections 3 to 9 (folder tree, extension counts, class table, sizes and modes, raw annotation samples, box format and statistics, duplicate counts, filename patterns), and the figures. Stage 2 will be written for the dataset's real structure.